# Laboratorio — Fuentes de Datos de Internet y Grafos en Redes Sociales

**Curso:** Minería de Datos (EIN132A25) · Universidad Técnica Federico Santa María
**Duración:** 2 horas

En este laboratorio aprenderás a **obtener datos desde la web** y a **modelar relaciones sociales como grafos** para descubrir estructura: usuarios influyentes, puentes entre grupos y comunidades.

## Objetivos

- Entender las tres dimensiones de la minería web (contenido, estructura, uso).
- Hacer **web scraping** responsable con `requests` + `BeautifulSoup`, respetando `robots.txt` y *rate limiting*.
- Construir grafos con **NetworkX** a partir de datos reales y simulados.
- Calcular **métricas de centralidad**: grado, intermediación (*betweenness*), cercanía (*closeness*) y coeficiente de *clustering*.
- Detectar **comunidades** con el algoritmo de **Louvain** y medir la **modularidad**.
- Visualizar e interpretar la estructura de una red social.

## Estructura

| Bloque | Tema |
|--------|------|
| 1 | Web scraping responsable (`quotes.toscrape.com`) |
| 2 | Construcción de grafos con NetworkX |
| 3 | Centralidad: ¿quién es importante en la red? |
| 4 | Detección de comunidades con Louvain |
| 5 | Caso real: red de *Game of Thrones* (dataset externo) |
| 6 | Ejercicios |


## 0. Setup — Dependencias

Si trabajas en tu máquina o en Google Colab, descomenta e instala lo necesario.

In [ ]:
# Descomenta si no las tienes instaladas:
# !pip install requests beautifulsoup4 pandas matplotlib networkx python-louvain


In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Louvain: el paquete se instala como 'python-louvain' pero se importa como 'community'
import community as community_louvain

plt.rcParams["figure.figsize"] = (9, 6)
print("Librerias cargadas correctamente")


---
## 1. Web Scraping responsable

La web es la fuente de datos más grande del mundo. Distinguimos tres tipos de minería web:

| Dimensión | Qué extrae | Ejemplo |
|-----------|-----------|---------|
| **Content mining** | Texto, imágenes, video | Reseñas, noticias, tweets |
| **Structure mining** | Topología de enlaces | Grafos, PageRank |
| **Usage mining** | Logs de navegación | Clickstreams |

Hoy combinamos **content + structure**: bajamos contenido y lo convertimos en un grafo.

### Reglas de oro (ética y legalidad)

| Hacer | No hacer |
|-------|----------|
| Revisar `robots.txt` | Ignorar restricciones del sitio |
| 1–2 s entre requests (*rate limiting*) | Lanzar cientos de requests/seg |
| Preferir **APIs oficiales** | Scrapear datos personales (emails, teléfonos) |
| Respetar los *Terms of Service* | Violar el ToS deliberadamente |

Usaremos **`quotes.toscrape.com`**, un sitio creado *explícitamente para practicar scraping* — sin riesgos legales ni claves de API.


### 1.1 Revisar `robots.txt` antes de scrapear

Siempre el primer paso: preguntar al sitio qué nos permite.

In [ ]:
from urllib.robotparser import RobotFileParser

BASE = "https://quotes.toscrape.com"
rp = RobotFileParser()
rp.set_url(f"{BASE}/robots.txt")
try:
    rp.read()
    permitido = rp.can_fetch("*", f"{BASE}/page/1/")
    print(f"Podemos scrapear {BASE}/page/1/?  ->  {permitido}")
except Exception as e:
    print("No se pudo leer robots.txt (sin internet?):", e)


### 1.2 Descargar y parsear una página

Inspeccionando el HTML del sitio vemos que cada cita está en un `<div class="quote">` con:
- el texto en `<span class="text">`
- el autor en `<small class="author">`
- las etiquetas en `<a class="tag">`

> 💡 **Nota:** El notebook incluye un *fallback* con datos de ejemplo embebidos. Si hay internet, scrapea en vivo; si no, usa la copia local para que el lab siempre funcione.

In [ ]:
# Datos de respaldo (fallback) por si no hay conexion en el aula.
# Provienen de las primeras paginas de quotes.toscrape.com.
FALLBACK = [
    {"text": "The world as we have created it is a process of our thinking.", "author": "Albert Einstein", "tags": ["change", "deep-thoughts", "thinking", "world"]},
    {"text": "It is our choices, Harry, that show what we truly are.", "author": "J.K. Rowling", "tags": ["abilities", "choices"]},
    {"text": "There are only two ways to live your life.", "author": "Albert Einstein", "tags": ["inspirational", "life", "live", "miracle", "miracles"]},
    {"text": "The person, be it gentleman or lady, who has not pleasure in a good novel.", "author": "Jane Austen", "tags": ["aliteracy", "books", "classic", "humor"]},
    {"text": "Imperfection is beauty, madness is genius.", "author": "Marilyn Monroe", "tags": ["be-yourself", "inspirational"]},
    {"text": "Try not to become a man of success. Rather become a man of value.", "author": "Albert Einstein", "tags": ["adulthood", "success", "value"]},
    {"text": "It is better to be hated for what you are than loved for what you are not.", "author": "Andre Gide", "tags": ["life", "love"]},
    {"text": "I have not failed. I've just found 10,000 ways that won't work.", "author": "Thomas A. Edison", "tags": ["edison", "failure", "inspirational", "paraphrased"]},
    {"text": "A woman is like a tea bag; you never know how strong it is until it's in hot water.", "author": "Eleanor Roosevelt", "tags": ["misattributed-eleanor-roosevelt"]},
    {"text": "A day without sunshine is like, you know, night.", "author": "Steve Martin", "tags": ["humor", "obvious", "simile"]},
    {"text": "This life is what you make it.", "author": "Marilyn Monroe", "tags": ["friends", "heartbreak", "inspirational", "life", "love", "sisters"]},
    {"text": "It takes a great deal of bravery to stand up to our enemies.", "author": "J.K. Rowling", "tags": ["courage", "friends"]},
    {"text": "If you can't explain it to a six year old, you don't understand it yourself.", "author": "Albert Einstein", "tags": ["simplicity", "understand"]},
    {"text": "You may not be her first, her last, or her only.", "author": "Bob Marley", "tags": ["love"]},
    {"text": "I like nonsense, it wakes up the brain cells.", "author": "Dr. Seuss", "tags": ["fantasy"]},
    {"text": "I may not have gone where I intended to go, but I think I ended up where I needed to be.", "author": "Douglas Adams", "tags": ["life", "navigation"]},
    {"text": "The opposite of love is not hate, it's indifference.", "author": "Elie Wiesel", "tags": ["activism", "apathy", "hate", "indifference", "inspirational", "love", "opposite", "philosophy"]},
    {"text": "It is not a lack of love, but a lack of friendship that makes unhappy marriages.", "author": "Friedrich Nietzsche", "tags": ["friendship", "lack-of-friendship", "love", "marriage", "unhappy-marriage"]},
    {"text": "Good friends, good books, and a sleepy conscience: this is the ideal life.", "author": "Mark Twain", "tags": ["books", "contentment", "friends", "friendship", "life"]},
    {"text": "Life is what happens to us while we are making other plans.", "author": "Allen Saunders", "tags": ["fate", "life", "misattributed-john-lennon", "planning", "plans"]},
]


def parse_quotes(html):
    """Extrae las citas de un HTML de quotes.toscrape.com."""
    soup = BeautifulSoup(html, "html.parser")
    filas = []
    for q in soup.find_all("div", class_="quote"):
        filas.append({
            "text":   q.find("span", class_="text").get_text(strip=True),
            "author": q.find("small", class_="author").get_text(strip=True),
            "tags":   [t.get_text(strip=True) for t in q.find_all("a", class_="tag")],
        })
    return filas


def scrape_pagina(n):
    """Descarga una pagina. Devuelve lista de citas o None si falla."""
    url = f"{BASE}/page/{n}/"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return parse_quotes(r.text)
    except Exception as e:
        print(f"  No se pudo descargar la pagina {n}: {e}")
        return None


# Probamos una pagina
prueba = scrape_pagina(1)
if prueba:
    print(f"Scraping en vivo: {len(prueba)} citas en la pagina 1")
else:
    print("Usaremos el dataset de respaldo (modo offline)")


### 1.3 Scraping de varias páginas (paginación)

El sitio tiene varias páginas (`/page/1/`, `/page/2/`, ...). Recorremos hasta que no haya más resultados, **esperando 1 segundo entre cada request** para no saturar el servidor.

In [ ]:
registros = []
USAR_FALLBACK = prueba is None

if not USAR_FALLBACK:
    for n in range(1, 11):           # como maximo 10 paginas
        citas = scrape_pagina(n)
        if not citas:                # pagina vacia -> terminamos
            break
        registros.extend(citas)
        print(f"Pagina {n}: {len(citas)} citas (acumulado: {len(registros)})")
        time.sleep(1)                # rate limiting
else:
    registros = FALLBACK
    print(f"Modo offline: {len(registros)} citas del dataset de respaldo")

df = pd.DataFrame(registros)
print(f"\nTotal de citas: {len(df)}")
df.head()


### 1.4 Guardar los datos

Buena práctica: persistir lo scrapeado en un CSV para no tener que volver a descargar.

In [ ]:
df_guardar = df.copy()
df_guardar["tags"] = df_guardar["tags"].apply(lambda ts: ", ".join(ts))
df_guardar.to_csv("quotes_scraped.csv", index=False)
print("Guardado en quotes_scraped.csv")

# Un poco de exploracion: autores y etiquetas mas frecuentes
print("\nAutores mas citados:")
print(df["author"].value_counts().head())

todas_tags = df.explode("tags")["tags"]
print("\nEtiquetas mas frecuentes:")
print(todas_tags.value_counts().head(10))


---
## 2. De datos a grafos con NetworkX

Un **grafo** es `G = (V, E)`: un conjunto de **nodos** (V) y **aristas** (E) que los conectan.

| Tipo | Ejemplo | Característica |
|------|---------|---------------|
| Dirigido | Twitter (seguir) | A→B ≠ B→A |
| No dirigido | Facebook (amistad) | A–B = B–A |
| Ponderado | Nº de interacciones | la arista tiene peso |

### 2.1 Construir un grafo desde los datos scrapeados

Vamos a conectar **autores que comparten etiquetas (temas)**. Si dos autores fueron citados con el mismo *tag*, trazamos una arista entre ellos, con peso = número de tags en común. Esto crea una **red de afinidad temática**.

In [ ]:
from itertools import combinations
from collections import defaultdict

# tag -> conjunto de autores que la usan
autores_por_tag = defaultdict(set)
for _, fila in df.iterrows():
    for t in fila["tags"]:
        autores_por_tag[t].add(fila["author"])

# Contamos tags compartidos entre cada par de autores
peso_par = defaultdict(int)
for tag, autores in autores_por_tag.items():
    for a, b in combinations(sorted(autores), 2):
        peso_par[(a, b)] += 1

G_autores = nx.Graph()
G_autores.add_nodes_from(df["author"].unique())
for (a, b), w in peso_par.items():
    G_autores.add_edge(a, b, weight=w)

print(f"Red de autores: {G_autores.number_of_nodes()} nodos, {G_autores.number_of_edges()} aristas")
print(f"Densidad: {nx.density(G_autores):.3f}")


In [ ]:
# Visualizacion rapida de la red de autores
plt.figure(figsize=(11, 8))
if G_autores.number_of_edges() > 0:
    pos = nx.spring_layout(G_autores, seed=42, k=0.6)
    pesos = [G_autores[u][v]["weight"] for u, v in G_autores.edges()]
else:
    pos = nx.spring_layout(G_autores, seed=42)
    pesos = 1.0

nx.draw_networkx_nodes(G_autores, pos, node_color="#4C9BE8", node_size=900, alpha=0.9)
nx.draw_networkx_edges(G_autores, pos, width=pesos, alpha=0.4)
nx.draw_networkx_labels(G_autores, pos, font_size=8)
plt.title("Red de autores conectados por temas compartidos")
plt.axis("off")
plt.tight_layout()
plt.show()


### 2.2 Un dataset clásico: el *Karate Club* de Zachary

La red de autores es pequeña. Para estudiar centralidad y comunidades usaremos también el **Zachary's Karate Club**: 34 miembros de un club de karate que terminó dividiéndose en dos por un conflicto. Es el "Hola Mundo" del análisis de redes y viene incluido en NetworkX.

In [ ]:
K = nx.karate_club_graph()
print(f"Karate Club: {K.number_of_nodes()} nodos, {K.number_of_edges()} aristas")
print(f"Densidad: {nx.density(K):.3f}")
print(f"Clustering promedio: {nx.average_clustering(K):.3f}")
print(f"Diametro (maxima distancia entre dos nodos): {nx.diameter(K)}")

plt.figure(figsize=(10, 7))
pos_k = nx.spring_layout(K, seed=42)
nx.draw_networkx(K, pos_k, node_color="#9BD18A", node_size=600, font_size=8, edge_color="gray")
plt.title("Zachary's Karate Club")
plt.axis("off")
plt.show()


---
## 3. Centralidad: ¿quién es importante en la red?

No todos los nodos son iguales. Las **métricas de centralidad** miden la importancia de cada nodo de distintas formas:

| Métrica | Pregunta que responde | Detecta |
|---------|----------------------|---------|
| **Grado** (*degree*) | ¿Cuántas conexiones directas tiene? | Populares / *influencers* |
| **Intermediación** (*betweenness*) | ¿Cuántos caminos más cortos pasan por él? | **Puentes** entre grupos |
| **Cercanía** (*closeness*) | ¿Qué tan cerca está de todos los demás? | Buenos difusores de info |
| **Clustering** | ¿Cuántos de sus vecinos se conocen entre sí? | Grupos cohesivos |


In [ ]:
deg = nx.degree_centrality(K)
btw = nx.betweenness_centrality(K)
clo = nx.closeness_centrality(K)
clu = nx.clustering(K)

central = pd.DataFrame({
    "grado": deg,
    "betweenness": btw,
    "closeness": clo,
    "clustering": clu,
}).sort_values("betweenness", ascending=False)

print("Top 5 nodos por intermediacion (betweenness):")
central.head().round(3)


In [ ]:
# Visualizar: tamano = grado, color = betweenness (los "puentes" resaltan)
plt.figure(figsize=(11, 8))
node_size = [3000 * deg[n] for n in K.nodes()]
node_color = [btw[n] for n in K.nodes()]

nodes = nx.draw_networkx_nodes(K, pos_k, node_size=node_size,
                               node_color=node_color, cmap="plasma")
nx.draw_networkx_edges(K, pos_k, alpha=0.3)
nx.draw_networkx_labels(K, pos_k, font_size=8)
plt.colorbar(nodes, label="Betweenness (intermediacion)")
plt.title("Tamano = grado | Color = intermediacion")
plt.axis("off")
plt.tight_layout()
plt.show()


> **Interpreta:** Los nodos **0** y **33** (los dos instructores que iniciaron el conflicto) tienen el mayor grado e intermediación: son los líderes. Los nodos con alto *betweenness* pero grado moderado son **puentes** — si los quitas, la red se fragmenta.

---
## 4. Detección de comunidades con Louvain

Una **comunidad** es un grupo de nodos densamente conectados entre sí, pero poco conectados con el resto.

El algoritmo de **Louvain** busca la partición que **maximiza la modularidad Q**, de forma rápida y escalable:

| Modularidad Q | Interpretación |
|---------------|----------------|
| ≤ 0 | Sin estructura de comunidades |
| ~0.3 | Comunidades débiles |
| ≥ 0.5 | Comunidades significativas |
| → 1 | Comunidades perfectas |


In [ ]:
particion = community_louvain.best_partition(K, random_state=42)
Q = community_louvain.modularity(particion, K)

n_com = len(set(particion.values()))
print(f"Comunidades detectadas: {n_com}")
print(f"Modularidad Q = {Q:.3f}")

# Quien quedo en cada comunidad?
for c in sorted(set(particion.values())):
    miembros = [n for n, com in particion.items() if com == c]
    print(f"  Comunidad {c}: {miembros}")


In [ ]:
# Colorear la red por comunidad
plt.figure(figsize=(11, 8))
colores = [particion[n] for n in K.nodes()]
nx.draw_networkx_nodes(K, pos_k, node_color=colores, cmap="tab10",
                       node_size=600, alpha=0.95)
nx.draw_networkx_edges(K, pos_k, alpha=0.3)
nx.draw_networkx_labels(K, pos_k, font_size=8)
plt.title(f"Comunidades (Louvain) - {n_com} grupos, Q = {Q:.3f}")
plt.axis("off")
plt.tight_layout()
plt.show()


### 4.1 Validación: red simulada con comunidades conocidas

Generamos una red **sintética** donde *sabemos* que hay 4 grupos (modelo *planted partition*: muchas aristas dentro de cada grupo, pocas entre grupos). Es como una red de Twitter con 4 "burbujas". Comprobamos si Louvain las recupera.

In [ ]:
# 4 comunidades de 12 nodos; p_in=0.45 (densa internamente), p_out=0.02 (poco entre grupos)
G_sim = nx.planted_partition_graph(l=4, k=12, p_in=0.45, p_out=0.02, seed=7)

part_sim = community_louvain.best_partition(G_sim, random_state=42)
Q_sim = community_louvain.modularity(part_sim, G_sim)
print(f"Comunidades detectadas: {len(set(part_sim.values()))} (esperadas: 4)")
print(f"Modularidad Q = {Q_sim:.3f}")

plt.figure(figsize=(11, 8))
pos_s = nx.spring_layout(G_sim, seed=7)
nx.draw_networkx_nodes(G_sim, pos_s, node_color=[part_sim[n] for n in G_sim.nodes()],
                       cmap="tab10", node_size=300)
nx.draw_networkx_edges(G_sim, pos_s, alpha=0.2)
plt.title(f"Red simulada - Louvain recupero {len(set(part_sim.values()))} comunidades (Q={Q_sim:.3f})")
plt.axis("off")
plt.tight_layout()
plt.show()


---
## 5. Caso real: red de personajes de *Game of Thrones*

Apliquemos todo el pipeline a un **dataset real y famoso** (popular en Kaggle): el *Network of Thrones*
de A. Beveridge & J. Shan, donde dos personajes están conectados si se mencionan cerca en el texto de
*A Storm of Swords*. El peso de la arista = número de co-ocurrencias.

Lo descargamos **directamente desde una URL pública** (GitHub *raw*, sin claves de API) con `pandas`.

> El CSV tiene tres columnas: `Source`, `Target`, `Weight`.

In [ ]:
URL_GOT = ("https://raw.githubusercontent.com/melaniewalsh/"
           "sample-social-network-datasets/master/sample-datasets/"
           "game-of-thrones/got-edges.csv")

# Fallback offline por si no hay internet (subconjunto representativo)
GOT_FALLBACK = [
    ("Robb","Catelyn",10),("Robb","Jon",12),("Robb","Bran",8),("Robb","Theon",9),
    ("Catelyn","Sansa",6),("Catelyn","Eddard",10),("Sansa","Arya",6),("Bran","Rickon",5),
    ("Jon","Samwell",10),("Jon","Mance",8),("Jon","Ygritte",9),("Eddard","Robert",12),
    ("Tyrion","Cersei",9),("Tyrion","Jaime",12),("Cersei","Jaime",14),("Tyrion","Tywin",10),
    ("Cersei","Joffrey",12),("Joffrey","Sansa",8),("Tyrion","Sansa",7),("Tyrion","Bronn",10),
    ("Tyrion","Shae",9),("Tywin","Jaime",8),("Robert","Cersei",8),("Joffrey","Tywin",6),
    ("Daenerys","Jorah",12),("Daenerys","Drogo",10),("Daenerys","Barristan",7),("Jorah","Barristan",5),
    ("Stannis","Davos",11),("Stannis","Melisandre",9),("Davos","Melisandre",6),("Robert","Stannis",6),
    ("Arya","Sandor",9),("Sandor","Sansa",6),("Sandor","Joffrey",5),("Eddard","Cersei",6),
    ("Jaime","Brienne",10),("Brienne","Catelyn",5),("Samwell","Mance",4),("Theon","Bran",5),
]

try:
    got = pd.read_csv(URL_GOT)
    print(f"Descargado en vivo: {len(got)} aristas")
except Exception as e:
    print("Sin internet, usando fallback offline:", e)
    got = pd.DataFrame(GOT_FALLBACK, columns=["Source", "Target", "Weight"])

got.head()


### 5.1 Construir el grafo ponderado

In [ ]:
GOT = nx.from_pandas_edgelist(got, "Source", "Target", edge_attr="Weight")

print(f"Personajes (nodos): {GOT.number_of_nodes()}")
print(f"Relaciones (aristas): {GOT.number_of_edges()}")
print(f"Densidad: {nx.density(GOT):.3f}")
print(f"Clustering promedio: {nx.average_clustering(GOT):.3f}")


### 5.2 ¿Quiénes son los personajes más centrales?

Usamos el **peso** de las aristas (las co-ocurrencias) en la intermediación.

In [ ]:
deg_got = nx.degree_centrality(GOT)
btw_got = nx.betweenness_centrality(GOT, weight="Weight")
# PageRank: otra medida de influencia (la que usa Google para páginas web)
pr_got = nx.pagerank(GOT, weight="Weight")

ranking = pd.DataFrame({
    "grado": deg_got,
    "betweenness": btw_got,
    "pagerank": pr_got,
}).sort_values("pagerank", ascending=False)

print("Top 10 personajes por PageRank:")
ranking.head(10).round(3)


### 5.3 Detectar las "casas" con Louvain

Las comunidades que encuentra Louvain deberían parecerse a las casas/facciones de la saga
(Stark, Lannister, la Guardia de la Noche, Daenerys en Essos, etc.).

In [ ]:
part_got = community_louvain.best_partition(GOT, weight="Weight", random_state=42)
Q_got = community_louvain.modularity(part_got, GOT, weight="Weight")

print(f"Comunidades detectadas: {len(set(part_got.values()))}")
print(f"Modularidad Q = {Q_got:.3f}\n")

for c in sorted(set(part_got.values())):
    miembros = [n for n, k in part_got.items() if k == c]
    print(f"Comunidad {c} ({len(miembros)} personajes): {', '.join(miembros[:12])}"
          + (" ..." if len(miembros) > 12 else ""))


In [ ]:
# Visualizacion: color = comunidad, tamano = PageRank
plt.figure(figsize=(14, 10))
pos_got = nx.spring_layout(GOT, weight="Weight", seed=42, k=0.5)

node_size = [8000 * pr_got[n] for n in GOT.nodes()]
node_color = [part_got[n] for n in GOT.nodes()]
edge_w = [0.3 * GOT[u][v]["Weight"] for u, v in GOT.edges()]

nx.draw_networkx_nodes(GOT, pos_got, node_size=node_size, node_color=node_color,
                       cmap="tab10", alpha=0.9)
nx.draw_networkx_edges(GOT, pos_got, width=edge_w, alpha=0.25)
# Etiquetar solo los personajes mas importantes
top_chars = sorted(pr_got, key=pr_got.get, reverse=True)[:15]
nx.draw_networkx_labels(GOT, pos_got, labels={n: n for n in top_chars}, font_size=9)

plt.title(f"Red de Game of Thrones - {len(set(part_got.values()))} comunidades (Q={Q_got:.3f})")
plt.axis("off")
plt.tight_layout()
plt.show()


> **Interpreta:** Los nodos grandes (alto PageRank) son los personajes que articulan la trama
(Tyrion, Jon, Daenerys, Sansa...). Los colores agrupan facciones que interactúan mucho entre sí.
Personajes con alto *betweenness* (p. ej. los que cruzan de una casa a otra) son **puentes**
narrativos: si los quitas, las tramas quedan desconectadas.

> 📦 **Otras fuentes de grafos** para experimentar (descarga directa por URL, sin login):
> - **SNAP** (Stanford): redes de Facebook, Twitter, colaboración científica — https://snap.stanford.edu/data/
> - **Network Repository**: https://networkrepository.com/
> - **Kaggle**: busca *"social network"* o *"graph"* (requiere cuenta para la API).

---
## 6. Síntesis

En este lab recorrimos un pipeline completo de minería web + redes sociales:

1. **Obtuvimos datos** de la web de forma responsable (`robots.txt`, *rate limiting*).
2. **Construimos grafos** a partir de datos scrapeados, simulados y de un **dataset real externo**.
3. **Medimos centralidad** (grado, betweenness, closeness, PageRank) para encontrar nodos influyentes y puentes.
4. **Detectamos comunidades** con Louvain y validamos con una red sintética y con la red de *Game of Thrones*.

Este flujo es la base de aplicaciones reales: detección de *influencers*, segmentación de audiencias, análisis de propagación de información y *echo chambers*.

---
## 7. Ejercicios

Completa las celdas en blanco. Apóyate en el código de arriba.

### Ejercicio 1 — Más datos, mejor red

Modifica el scraping para recolectar también las páginas restantes (o usa el `df` completo). Construye la red de autores y responde: ¿cuál es el autor con **mayor grado** (más conexiones temáticas)? ¿Y el de mayor **betweenness**?

### Ejercicio 2 — Top influencers

Para el grafo del *Karate Club*, crea una tabla con los **5 nodos más centrales** según cada métrica (grado, betweenness, closeness). ¿Coinciden los rankings? ¿Qué nodo es un "puente" claro (alto betweenness pero grado no tan alto)?

### Ejercicio 3 — Sensibilidad de las comunidades

En la red simulada `planted_partition_graph`, sube `p_out` de `0.02` a `0.15` (más conexiones entre grupos). Vuelve a correr Louvain. ¿Cómo cambian el número de comunidades y la modularidad Q? Explica por qué.

### Ejercicio 4 — Puentes en Game of Thrones

Sobre el grafo `GOT`, calcula el **betweenness centrality ponderado** y muestra el top 5. ¿Qué personaje es el mayor "puente" entre comunidades? Compáralo con el ranking de PageRank: ¿el personaje más influyente es también el mayor puente? Explica la diferencia entre ambas métricas.

### Desafío final — Tu propia red

Elige **otra fuente** (otra página de `*.toscrape.com`, un dataset de SNAP/Network Repository, o un CSV de Kaggle) y repite el pipeline completo: datos → grafo → centralidad → comunidades. Entrega un breve párrafo interpretando: ¿quiénes son los nodos clave y qué representan las comunidades encontradas?

### Recursos

- **NetworkX** — documentación: https://networkx.org/documentation/stable/
- **python-louvain** (community): https://python-louvain.readthedocs.io/
- **BeautifulSoup**: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Sitios para practicar scraping: https://quotes.toscrape.com · https://books.toscrape.com
- M. Newman, *Networks* (2018) — referencia teórica de análisis de redes.
